# 03 — Análisis Predictivo y Sistema de Recomendación

**Editorial ficticia**: Historia & Biografía
**Fuente de datos**: `src/data/books_clean.csv`

Dos objetivos:
1. **Modelo predictivo**: ¿qué características del libro predicen un rating alto (≥ 4.0)?
2. **Sistema de recomendación**: función `recommend(book_title, n)` basada en TF-IDF + similitud coseno.

In [ ]:
import os
import ast
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import classification_report
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (10, 5)})
sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
CSV_PATH = os.path.join("..", "src", "data", "books_clean.csv")

df = pd.read_csv(CSV_PATH)

df["genres"] = df["genres"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) and x.startswith("[") else []
)

# Solo libros con valoraciones reales
df = df[df["has_ratings"].eq(True) & df["average_rating"].notna()].reset_index(drop=True)
print(f"Libros para modelado: {len(df):,}")
df.head(3)

## 1. Modelo de éxito editorial

### 1.1 Ingeniería de características

In [ ]:
# ── Target: clasificación binaria ───────────────────────────
df["high_rating"] = (df["average_rating"] >= 4.0).astype(int)
print(f"Distribución del target:")
print(df["high_rating"].value_counts().rename({0: "rating < 4.0", 1: "rating >= 4.0"}).to_string())

# ── Features numéricas ───────────────────────────────────────
df["log_num_ratings"] = np.log1p(df["num_ratings"].fillna(0).astype(float))

num_features = df[["num_pages", "year_published", "log_num_ratings"]].copy()
num_features["num_pages"]      = num_features["num_pages"].fillna(num_features["num_pages"].median())
num_features["year_published"] = num_features["year_published"].fillna(num_features["year_published"].median())

# ── Frecuencia del autor (target-free encoding) ───────────────
author_freq = df["author"].value_counts()
num_features["author_freq"] = df["author"].map(author_freq).fillna(1)

# ── Top-50 géneros como columnas binarias ────────────────────
from collections import Counter
top50_genres = [g for g, _ in Counter(
    genre for lst in df["genres"] for genre in lst
).most_common(50)]

mlb = MultiLabelBinarizer(classes=top50_genres)
genre_matrix = pd.DataFrame(
    mlb.fit_transform(df["genres"]),
    columns=[f"genre_{g.replace(' ', '_')}" for g in top50_genres]
)

X = pd.concat([num_features.reset_index(drop=True), genre_matrix], axis=1)
y = df["high_rating"]

print(f"\nShape del feature matrix: {X.shape}")

### 1.2 Entrenamiento y validación (5-fold CV)

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

models = {
    "RandomForest": RandomForestClassifier(
        n_estimators=200, max_depth=12, class_weight="balanced",
        random_state=RANDOM_SEED, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=(y == 0).sum() / (y == 1).sum(),
        random_state=RANDOM_SEED, eval_metric="logloss",
        verbosity=0
    ),
}

results = {}
for name, clf in models.items():
    scores = cross_validate(
        clf, X, y, cv=cv,
        scoring=["accuracy", "f1", "roc_auc"],
        return_train_score=False
    )
    results[name] = {
        "Accuracy":  f"{scores['test_accuracy'].mean():.3f} ± {scores['test_accuracy'].std():.3f}",
        "F1":        f"{scores['test_f1'].mean():.3f} ± {scores['test_f1'].std():.3f}",
        "ROC-AUC":   f"{scores['test_roc_auc'].mean():.3f} ± {scores['test_roc_auc'].std():.3f}",
    }

results_df = pd.DataFrame(results).T
print("=== Resultados 5-fold CV ===")
print(results_df.to_string())

### 1.3 Feature importance

In [ ]:
# Entrenar RandomForest en el conjunto completo para feature importance
rf = RandomForestClassifier(
    n_estimators=200, max_depth=12, class_weight="balanced",
    random_state=RANDOM_SEED, n_jobs=-1
)
rf.fit(X, y)

feat_imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False).head(20)

fig, ax = plt.subplots(figsize=(10, 6))
feat_imp.sort_values().plot(kind="barh", ax=ax, color="cornflowerblue", edgecolor="white")
ax.set(title="Top 20 features más importantes (Random Forest)", xlabel="Importancia")
plt.tight_layout()
plt.show()

print("Top 10 features:")
print(feat_imp.head(10).to_string())

### 1.4 Conclusiones para la editorial

- **`log_num_ratings`** suele ser el predictor más potente: los libros ya populares tienden a recibir mejores valoraciones medias (efecto reputación).
- **`year_published`** captura tendencias temporales: los libros recientes pueden beneficiarse de un efecto novedad.
- Los géneros con mayor peso positivo en feature importance señalan los nichos donde el corpus es más exigente y donde los libros tienden a destacar.
- La editorial debería enfocarse en manuscritos que pertenezcan a los géneros de alta importancia y de autores con historial probado en el corpus.

## 2. Sistema de recomendación

### 2.1 Construcción del índice TF-IDF

In [ ]:
# Corpus: descripción + géneros concatenados
df["text_corpus"] = (
    df["book_description"].fillna("") + " " +
    df["genres"].apply(lambda g: " ".join(g))
)

tfidf_vec = TfidfVectorizer(
    max_features=8000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)
tfidf_matrix = tfidf_vec.fit_transform(df["text_corpus"])

print(f"Matriz TF-IDF: {tfidf_matrix.shape}  ({tfidf_matrix.nnz:,} valores no nulos)")

In [ ]:
def recommend(book_title: str, n: int = 5, query_text: str = None) -> pd.DataFrame:
    """Devuelve los n libros más similares al título dado.

    Parámetros
    ----------
    book_title  : Título exacto (case-insensitive). Si no está en el corpus,
                  se usa query_text (cold-start).
    n           : Número de recomendaciones.
    query_text  : Descripción libre del libro externo (cold-start).

    Retorna
    -------
    DataFrame con columnas: book_title, author, average_rating, genres.
    """
    mask = df["book_title"].str.lower() == book_title.lower()

    if mask.any():
        idx       = df[mask].index[0]
        query_vec = tfidf_matrix[idx]
        exclude   = {idx}
    elif query_text:
        print(f"[INFO] '{book_title}' no está en el corpus — usando cold-start con query_text.")
        query_vec = tfidf_vec.transform([query_text])
        exclude   = set()
    else:
        raise ValueError(
            f"'{book_title}' no encontrado en el corpus. "
            "Proporciona query_text para manejar el caso cold-start."
        )

    scores  = cosine_similarity(query_vec, tfidf_matrix).flatten()
    top_idx = [
        i for i in scores.argsort()[::-1]
        if i not in exclude
    ][:n]

    result = df.iloc[top_idx][["book_title", "author", "average_rating", "genres"]].copy()
    result.insert(0, "similarity", scores[top_idx].round(4))
    return result.reset_index(drop=True)

### 2.2 Pruebas del recomendador

#### Caso a) Libro del dataset — *Dead Wake* (Erik Larson)

In [ ]:
print("=== Recomendaciones para 'Dead Wake' (está en el corpus) ===\n")
try:
    recs = recommend("Dead Wake: The Last Crossing of the Lusitania", n=5)
    print(recs[["similarity", "book_title", "author", "average_rating"]].to_string(index=False))
except ValueError as e:
    # Intentar con título parcial
    matches = df[df["book_title"].str.contains("Dead Wake", case=False)]
    print("Títulos que contienen 'Dead Wake':")
    print(matches["book_title"].values)

#### Caso b) Cold-start — *Fourth Wing* de Rebecca Yarros (Alas de Sangre)

No está en el corpus (es fantasy, no historia). Usamos su sinopsis para buscar libros similares en el dataset.

In [ ]:
FOURTH_WING_TEXT = (
    "Violet Sorrengail is ordered to join the riders quadrant and attempt to bond a dragon "
    "at Basgiath War College. The dragons are terrifying and unpredictable, and if they don't kill "
    "her, the other cadets might. War is coming and the stakes are high in this romantic fantasy "
    "epic set in a world of war, dragons, and dangerous secrets. A young woman must survive elite "
    "military training and navigate deadly political intrigue while powerful forces seek to destroy her."
)

print("=== Cold-start: Fourth Wing (Rebecca Yarros) ===")
print("Libros más similares en nuestro corpus de Historia/Biografía:\n")
recs_fw = recommend("Fourth Wing", n=5, query_text=FOURTH_WING_TEXT)
print(recs_fw[["similarity", "book_title", "author", "average_rating"]].to_string(index=False))
print()
print("[NOTA] Al ser fantasy, la similitud semántica con Historia/Biografía es baja.")
print("La función devuelve los más cercanos disponibles, pero la editorial debería")
print("ampliar el corpus con listas de fantasy para recomendaciones de nicho cruzado.")

#### Caso c) Cold-start — *Frankenstein* de Mary Shelley

In [ ]:
FRANKENSTEIN_TEXT = (
    "Victor Frankenstein, a brilliant scientist, becomes obsessed with creating life. "
    "He assembles a creature from dead body parts and animates it with electricity. "
    "The creature, intelligent and sensitive but hideously ugly, is abandoned by his creator "
    "and rejected by humanity. Seeking revenge, he pursues Frankenstein across Europe. "
    "A gothic horror novel exploring themes of ambition, responsibility, and what it means to be human."
)

print("=== Cold-start: Frankenstein (Mary Shelley) ===\n")
recs_frank = recommend("Frankenstein", n=5, query_text=FRANKENSTEIN_TEXT)
print(recs_frank[["similarity", "book_title", "author", "average_rating"]].to_string(index=False))

#### Caso d) Cold-start — *Antigone* de Sófocles

In [ ]:
ANTIGONE_TEXT = (
    "Antigone, daughter of Oedipus, defies the decree of King Creon of Thebes and attempts "
    "to bury her slain brother Polynices. Creon condemns her to death for disobeying state law. "
    "The tragedy explores the conflict between divine law and human authority, loyalty to family "
    "versus duty to the state, and the consequences of pride and hubris in ancient Greece."
)

print("=== Cold-start: Antigone (Sófocles) ===\n")
recs_antig = recommend("Antigone", n=5, query_text=ANTIGONE_TEXT)
print(recs_antig[["similarity", "book_title", "author", "average_rating"]].to_string(index=False))

### 2.3 Manejo del cold-start

Cuando un libro **no está en el corpus**, la función `recommend()` acepta un parámetro `query_text`:
- El texto se transforma con el mismo vectorizador TF-IDF ya entrenado.
- Se calcula similitud coseno contra todos los libros del dataset.
- Se devuelven los `n` más cercanos.

**Limitaciones**:
- Si el libro externo pertenece a un género muy diferente (fantasy, horror) los resultados tendrán baja similitud (< 0.1) y pueden no ser editorialmente relevantes.
- **Solución a largo plazo**: enriquecer el corpus con listas de géneros mixtos o usar modelos de embedding semántico (`sentence-transformers`) que capturan similitud conceptual más allá del vocabulario compartido.